I'll poke at networkx and see what happens if I naively (since I have no clue what I'm doing) try to bring the RDF graph into networkx.

In [2]:
from pathlib import Path
from rdflib import Graph

rdfpath = Path("../data/sidebar_graph.ttl")
g = Graph()
g.parse(rdfpath)
print(f"{len(g):,}")



352,428


In [3]:
from rdflib.extras.external_graph_libs import rdflib_to_networkx_graph

nxg = rdflib_to_networkx_graph(g)
print(f"Nodes: {len(nxg.nodes):,}")


Nodes: 198,459


In [4]:
node_list = [n for n in nxg.nodes]
from pprint import pprint
n = node_list[0]
print(f"type: {type(n)}")
print(f"__str__(): {n}")


type: <class 'rdflib.term.URIRef'>
__str__(): https://itiner-e.org/route-segment/23342


In [5]:
edge_list = [e for e in nxg.edges]
e = edge_list[0]
print(f"type: {type(n)}")
print(f"__str__(): {n}")

type: <class 'rdflib.term.URIRef'>
__str__(): https://itiner-e.org/route-segment/23342


In [6]:
neighbors = [neb for neb in nxg.neighbors(n)]
for neb in neighbors:
    print(neb)

Conjectured Secondary Road
https://pleiades.stoa.org/places/207066
https://pleiades.stoa.org/places/207468
https://purl.org/geojson/vocab#Feature
23342 Apulum-Straja-Obreja
https://pleiades.stoa.org/places/207316
https://pleiades.stoa.org/places/207038
https://pleiades.stoa.org/places/207408
https://pleiades.stoa.org/places/206927
https://pleiades.stoa.org/places/207422
https://pleiades.stoa.org/places/207284


In [7]:
print(n)
print(len(neighbors))
for neb in neighbors:
    edge_d = nxg.get_edge_data(n, neb)
    pprint(edge_d)

https://itiner-e.org/route-segment/23342
11
{'triples': [(rdflib.term.URIRef('https://itiner-e.org/route-segment/23342'),
              rdflib.term.URIRef('https://schema.org/description'),
              rdflib.term.Literal('Conjectured Secondary Road', lang='und'))],
 'weight': 1}
{'triples': [(rdflib.term.URIRef('https://itiner-e.org/route-segment/23342'),
              rdflib.term.URIRef('http://www.w3.org/2004/02/skos/core#relatedMatch'),
              rdflib.term.URIRef('https://pleiades.stoa.org/places/207066'))],
 'weight': 1}
{'triples': [(rdflib.term.URIRef('https://itiner-e.org/route-segment/23342'),
              rdflib.term.URIRef('http://www.w3.org/2004/02/skos/core#relatedMatch'),
              rdflib.term.URIRef('https://pleiades.stoa.org/places/207468'))],
 'weight': 1}
{'triples': [(rdflib.term.URIRef('https://itiner-e.org/route-segment/23342'),
              rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'),
              rdflib.term.URIRef('https:

OK, so all the triples that connect two nodes are just banged into a single edge between them as an object. Since my primary interest now is trying to develop clusters of close matches that eventually point to Pleiades places, maybe I can process the RDF to get a single-mode, directional network to play with in networkx.

Will try doing SPARQL query to get just those triples that contain closeMatch 

In [3]:
query = """
SELECT DISTINCT ?s ?o
WHERE {
    ?s skos:closeMatch ?o
}
"""
results = g.query(query)
for i, row in enumerate(results):
    if i >= 10:
        break
    print(f"{row.s} closely matches {row.o}") # type: ignore


http://nomisma.org/id/agathopolis closely matches http://dbpedia.org/resource/Agathopolis
http://nomisma.org/id/agathopolis closely matches http://vocab.getty.edu/tgn/7017007
http://nomisma.org/id/agathopolis closely matches https://ikmk.smb.museum/ndp/ort/1948
http://nomisma.org/id/agathopolis closely matches https://pleiades.stoa.org/places/216721
http://www.wikidata.org/entity/Q341027 closely matches https://pleiades.stoa.org/places/216721
http://nomisma.org/id/alexandria_on_the_caucasus closely matches http://dbpedia.org/resource/Begram
http://nomisma.org/id/alexandria_on_the_caucasus closely matches https://pleiades.stoa.org/places/59668
https://vici.org/vici/10850 closely matches https://pleiades.stoa.org/places/59668
http://www.wikidata.org/entity/Q2005166 closely matches https://pleiades.stoa.org/places/59668
http://nomisma.org/id/alia closely matches http://collection.britishmuseum.org/id/place/x47036


In [20]:
import logging
import networkx as nx
from pprint import pformat
from rdflib import URIRef, RDFS, SDO

class GazetteerResource:
    def __init__(self):
        self.uri = ""
        self.title = ""
        self.description = ""
        self.logger = logging.getLogger("GazetteerResource")

    def from_graph(self, g: Graph, uri_ref: URIRef):
        self.uri = str(uri_ref)
        for attr_name, predicate in  {"title": RDFS.label, "description": SDO.description}.items():
            values = [str(o) for s, p, o  in g.triples((uri_ref, predicate, None))]
            match len(values):
                case 0:
                    pass
                case 1:
                    setattr(self, attr_name, values[0])
                case _:
                    self.logger.warning(f"Unexpected number of values ({len(values)}) for attribute '{attr_name}' on {self.uri}:\n{pformat(values, indent=2)}\nUsing the first one.")
                    setattr(self, attr_name, values[0])



nxg = nx.Graph() # not directional
nxg.clear()
first_edge_node_ids = None
print(f"number of results rows: {len(results)}")
for i, row in enumerate(results):
    # add nodes for subjects and objects with attributes
    # NB: we add new nodes/edges and NetworkX quietly ignores any that are already present.
    uri_refs = [row.s, row.o]# type: ignore
    node_uris = []
    for uri_ref in uri_refs: 
        gr = GazetteerResource()
        gr.from_graph(g, uri_ref) # type: ignore 
        nxg.add_node(gr.uri, title=gr.title, description=gr.description)
        node_uris.append(gr.uri)
    try:
        this_edge = nxg.edges[*node_uris]
    except KeyError:
        nxg.add_edge(*node_uris, weight=1)
    else:
        this_edge["weight"] += 1
    
print(f"nxg nodes: {nxg.number_of_nodes()}")
print(f"nxg edges: {nxg.number_of_edges()}")
weight_info = dict()
for (u, v, wt) in nxg.edges.data('weight'):
    try:
        weight_info[wt]
    except KeyError:
        weight_info[wt] = 1
    else:
        weight_info[wt] += 1
from pprint import pprint
pprint(weight_info, indent=2)

Unexpected number of values (2) for attribute 'description' on http://www.wikidata.org/entity/Q217265:
['Historical region of Anatolia in Turkey', 'Historical region of Asia Minor']
Using the first one.


number of results rows: 115521


Unexpected number of values (2) for attribute 'description' on http://www.wikidata.org/entity/Q1034750:
['Cape in Italy', 'Head in Italy']
Using the first one.
Unexpected number of values (2) for attribute 'description' on http://www.wikidata.org/entity/Q1034750:
['Cape in Italy', 'Head in Italy']
Using the first one.
Unexpected number of values (2) for attribute 'description' on http://www.wikidata.org/entity/Q1034750:
['Cape in Italy', 'Head in Italy']
Using the first one.
Unexpected number of values (2) for attribute 'description' on http://www.wikidata.org/entity/Q217265:
['Historical region of Anatolia in Turkey', 'Historical region of Asia Minor']
Using the first one.
Unexpected number of values (2) for attribute 'description' on http://www.wikidata.org/entity/Q217265:
['Historical region of Anatolia in Turkey', 'Historical region of Asia Minor']
Using the first one.
Unexpected number of values (2) for attribute 'description' on http://www.wikidata.org/entity/Q217265:
['Historica

nxg nodes: 101298
nxg edges: 115176
{1: 114831, 2: 345}


In [26]:
pleiades_query = """
SELECT DISTINCT ?s ?o
WHERE {
    ?s skos:relatedMatch ?o
    FILTER(regex(str(?s), "pleiades.stoa.org"))
}
"""
pleiades_results = g.query(pleiades_query)
print(len(pleiades_results))
for i, row in enumerate(pleiades_results):
    uri_refs = [row.s, row.o]# type: ignore
    node_uris = []
    for uri_ref in uri_refs: 
        gr = GazetteerResource()
        gr.from_graph(g, uri_ref) # type: ignore 
        nxg.add_node(gr.uri, title=gr.title, description=gr.description)
        node_uris.append(gr.uri)
    try:
        this_edge = nxg.edges[*node_uris]
    except KeyError:
        nxg.add_edge(*node_uris, weight=1)
    else:
        this_edge["weight"] += 1
    
print(f"nxg nodes: {nxg.number_of_nodes()}")
print(f"nxg edges: {nxg.number_of_edges()}")
weight_info = dict()
for (u, v, wt) in nxg.edges.data('weight'):
    try:
        weight_info[wt]
    except KeyError:
        weight_info[wt] = 1
    else:
        weight_info[wt] += 1
from pprint import pprint
pprint(weight_info, indent=2)

Unexpected number of values (2) for attribute 'description' on http://www.wikidata.org/entity/Q1034750:
['Cape in Italy', 'Head in Italy']
Using the first one.
Unexpected number of values (2) for attribute 'description' on http://www.wikidata.org/entity/Q217265:
['Historical region of Anatolia in Turkey', 'Historical region of Asia Minor']
Using the first one.


19560
nxg nodes: 101484
nxg edges: 115354
{1: 95627, 2: 19727}
